# User Feedback
We can add `UserProxyAgent` to the team to provide user feedback during a run. See `Human-in-the-Loop` for more details about `UserProxyAgent`. <br>

To use the `UserProxyAgent` in the web search example, we simply add it to the team and update the selector function to always check for user feedback after the planning agent speaks. If the user responds with `"APPROVE"`, the conversation continues, otherwise, the planning agent tries again, until the user approves.

#### ***Reference URL***
https://microsoft.github.io/autogen/stable//user-guide/agentchat-user-guide/selector-group-chat.html#user-feedback

In [1]:
import asyncio
from autogen_ext.models.openai import OpenAIChatCompletionClient
from autogen_agentchat.agents import AssistantAgent
from autogen_agentchat.teams import SelectorGroupChat
from autogen_agentchat.messages import TextMessage
from autogen_agentchat.conditions import TextMentionTermination,MaxMessageTermination

In [2]:
import os
from dotenv import load_dotenv

# Load API key
load_dotenv()
api_key= os.getenv('OPENAI_API_KEY')

In [5]:
# Model client

model_client= OpenAIChatCompletionClient(model='gpt-4o', api_key=api_key)
model_client

In [6]:
# Planning Assistant Agent

planning_agent= AssistantAgent(
    name="PlanningAgent",
    description="An agent for planning tasks, this agent should be the first to engage when given a new task.",
    system_message="""
    You are a planning agent.
    Your job is to break down complex tasks into smaller, manageable subtasks.
    Your team members are:
        WebSearchAgent: Searches for information
        DataAnalystAgent: Performs calculations

    You only plan and delegate tasks - you do not execute them yourself.

    When assigning tasks, use this format:
    1. <agent> : <task>

    After all tasks are complete, summarize the findings and end with "TERMINATE".
    """,
    model_client=model_client
)

In [7]:
# Search Web Tool

def search_web_tool(query: str) -> str:
    # Simulate a web search
    if "2006-2007" in query:
        return """Here are the total points scored by Miami Heat players in the 2006-2007 season:
        Udonis Haslem: 844 points
        Dwayne Wade: 1397 points
        James Posey: 550 points
        ...
        """
    elif "2007-2008" in query:
        return "The number of total rebounds for Dwayne Wade in the Miami Heat season 2007-2008 is 214."
    elif "2008-2009" in query:
        return "The number of total rebounds for Dwayne Wade in the Miami Heat season 2008-2009 is 398."
    return "No data found."

In [8]:
# Web Search Assistant Agent

web_search_agent= AssistantAgent(
    name="WebSearchAgent",
    description="An agent for searching the web for information.",
    system_message="""
        You are a web search agent.
        Your only tool is search_web_tool - use it to find the information you need.
        You make only one search call at a time.   
        Once you have the results, you never do calculations or data analysis on them.
    """,
    model_client=model_client,
    tools= [search_web_tool],
    reflect_on_tool_use= False
    
)

In [9]:
def percentage_change_tool(start:float, end:float) -> float:
    '''Calculate percentage change'''
    
    if start == 0:
        return 0
    return ((end - start) / start) * 100

In [10]:
# Data Analyst Assistant Agent

data_analyst_agent= AssistantAgent(
    name="DataAnalystAgent",
    description="An agent for performing calculations and data analysis.",
    system_message="""
        You are a data analyst agent.
        Given the tasks you have been assigned, you should analyze the data and provide results using the tools provided.
        If you have not seen the data, ask for it.
    """,
    model_client=model_client,
    tools= [percentage_change_tool]
)

In [11]:
# Termination Condition
from autogen_agentchat.conditions import TextMentionTermination, MaxMessageTermination

text_mention_termination= TextMentionTermination(text="TERMINATE")
max_message_termination= MaxMessageTermination(max_messages=20)

combined_termination= text_mention_termination | max_message_termination

In [12]:
selector_prompt='''

Select an agent to perform the task.

{roles}

current conversation history :
{history}

Read the above conversation, then select an agent from {participants} to perform the next task.
Make sure that the planning agent has assigned task before other agents start working.
Only select one agent.
'''

To use the `UserProxyAgent` in the web search example, we simply add it to the team and update the selector function to always check for user feedback after the planning agent speaks. If the user responds with **"APPROVE"**, the conversation continues, otherwise, the planning agent tries again, until the user approves.

In [ ]:
from autogen_agentchat.agents import UserProxyAgent
from autogen_agentchat.messages import BaseAgentEvent, BaseChatMessage
from typing import Sequence

user_proxy_agent = UserProxyAgent("UserProxyAgent", description="A proxy for the user to approve or disapprove tasks.")


def selector_func_with_user_proxy(messages: Sequence[BaseAgentEvent | BaseChatMessage]) -> str | None:
    if messages[-1].source != planning_agent.name and messages[-1].source != user_proxy_agent.name:
        # Planning agent should be the first to engage when given a new task, or check progress.
        return planning_agent.name
    if messages[-1].source == planning_agent.name:
        if messages[-2].source == user_proxy_agent.name and "APPROVE" in messages[-1].content.upper():  # type: ignore
            # User has approved the plan, proceed to the next agent.
            return None
        # Use the user proxy agent to get the user's approval to proceed.
        return user_proxy_agent.name
    if messages[-1].source == user_proxy_agent.name:
        # If the user does not approve, return to the planning agent.
        if "APPROVE" not in messages[-1].content.upper():  # type: ignore
            return planning_agent.name
    return None


# Reset the previous agents and run the chat again with the user proxy agent and selector function.
# await team.reset()

team = SelectorGroupChat(
    [planning_agent, web_search_agent, data_analyst_agent, user_proxy_agent],
    model_client=model_client,
    termination_condition=combined_termination,
    selector_prompt=selector_prompt,
    selector_func=selector_func_with_user_proxy,
    allow_repeated_speaker=True,
)

In [15]:
task= "Who was the Miami Heat player with the highest point in the 2006-2007 season, and what was the percentage change in his total rebounds between the 2007-2008 and 2008-2009 seasons?"
task

'Who was the Miami Heat player with the highest point in the 2006-2007 season, and what was the percentage change in his total rebounds between the 2007-2008 and 2008-2009 seasons?'

In [16]:
from autogen_agentchat.ui import Console

await Console(team.run_stream(task=task))

---------- TextMessage (user) ----------
Who was the Miami Heat player with the highest point in the 2006-2007 season, and what was the percentage change in his total rebounds between the 2007-2008 and 2008-2009 seasons?
---------- TextMessage (PlanningAgent) ----------
To answer this question, we need to follow these steps:

1. Identify the Miami Heat player with the highest points in the 2006-2007 season.
2. Find the player's total rebounds for the 2007-2008 season.
3. Find the player's total rebounds for the 2008-2009 season.
4. Calculate the percentage change in total rebounds between the 2007-2008 and 2008-2009 seasons.

Let's delegate these tasks accordingly:

1. WebSearchAgent: Find out who was the Miami Heat player with the highest points during the 2006-2007 season.
2. WebSearchAgent: Find the total rebounds of that player in the 2007-2008 season.
3. WebSearchAgent: Find the total rebounds of that player in the 2008-2009 season.
4. DataAnalystAgent: Calculate the percentage ch

TaskResult(messages=[TextMessage(id='e308cb31-b210-4768-b847-2b72b99892d8', source='user', models_usage=None, metadata={}, created_at=datetime.datetime(2025, 7, 20, 8, 9, 14, 557151, tzinfo=datetime.timezone.utc), content='Who was the Miami Heat player with the highest point in the 2006-2007 season, and what was the percentage change in his total rebounds between the 2007-2008 and 2008-2009 seasons?', type='TextMessage'), TextMessage(id='372912a4-dad6-4eee-80f3-0c86cb8d383f', source='PlanningAgent', models_usage=RequestUsage(prompt_tokens=161, completion_tokens=208), metadata={}, created_at=datetime.datetime(2025, 7, 20, 8, 9, 18, 8241, tzinfo=datetime.timezone.utc), content="To answer this question, we need to follow these steps:\n\n1. Identify the Miami Heat player with the highest points in the 2006-2007 season.\n2. Find the player's total rebounds for the 2007-2008 season.\n3. Find the player's total rebounds for the 2008-2009 season.\n4. Calculate the percentage change in total re

#### Structured Output

In [ ]:
TaskResult(
    messages=[
        TextMessage(id='e308cb31-b210-4768-b847-2b72b99892d8', 
                    source='user', models_usage=None, metadata={}, created_at=datetime.datetime(2025, 7, 20, 8, 9, 14, 557151, tzinfo=datetime.timezone.utc), 
                    content='Who was the Miami Heat player with the highest point in the 2006-2007 season, and what was the percentage change in his total rebounds between the 2007-2008 and 2008-2009 seasons?', type='TextMessage'), 
                    
        TextMessage(id='372912a4-dad6-4eee-80f3-0c86cb8d383f', 
                    source='PlanningAgent', models_usage=RequestUsage(prompt_tokens=161, completion_tokens=208), metadata={}, created_at=datetime.datetime(2025, 7, 20, 8, 9, 18, 8241, tzinfo=datetime.timezone.utc), 
                    content="To answer this question, we need to follow these steps:\n\n1. Identify the Miami Heat player with the highest points in the 2006-2007 season.\n2. Find the player's total rebounds for the 2007-2008 season.\n3. Find the player's total rebounds for the 2008-2009 season.\n4. Calculate the percentage change in total rebounds between the 2007-2008 and 2008-2009 seasons.\n\nLet's delegate these tasks accordingly:\n\n1. WebSearchAgent: Find out who was the Miami Heat player with the highest points during the 2006-2007 season.\n2. WebSearchAgent: Find the total rebounds of that player in the 2007-2008 season.\n3. WebSearchAgent: Find the total rebounds of that player in the 2008-2009 season.\n4. DataAnalystAgent: Calculate the percentage change in the player's total rebounds from the 2007-2008 season to the 2008-2009 season.", type='TextMessage'), 
                    
        UserInputRequestedEvent(id='e2a335e2-8a52-475d-8eeb-f8ded01ae865', 
                                source='UserProxyAgent', models_usage=None, metadata={}, created_at=datetime.datetime(2025, 7, 20, 8, 9, 18, 10242, tzinfo=datetime.timezone.utc), request_id='6f878105-0de3-4a67-9fcd-c6df71b53d75', 
                                content='', type='UserInputRequestedEvent'), 
                                
        TextMessage(id='61ec0e1b-f817-494b-817d-fa4bf8728dc1', 
                    source='UserProxyAgent', models_usage=None, metadata={}, created_at=datetime.datetime(2025, 7, 20, 8, 9, 43, 178053, tzinfo=datetime.timezone.utc), 
                    content='APPROVE', type='TextMessage'), 
                    
        ToolCallRequestEvent(id='35f1f4d8-3306-409b-9d43-274114d16b66', 
                             source='WebSearchAgent', models_usage=RequestUsage(prompt_tokens=375, completion_tokens=26), metadata={}, created_at=datetime.datetime(2025, 7, 20, 8, 9, 47, 274215, tzinfo=datetime.timezone.utc), 
                             content=[FunctionCall(id='call_d5arZSv1xZvsB0HWjKMJdi8o', arguments='{"query":"Miami Heat highest points player 2006-2007 season"}', name='search_web_tool')], type='ToolCallRequestEvent'), 
                             
        ToolCallExecutionEvent(id='cd47187f-22b1-48f0-a5e3-033574ab09c8', source='WebSearchAgent', models_usage=None, metadata={}, created_at=datetime.datetime(2025, 7, 20, 8, 9, 47, 277218, tzinfo=datetime.timezone.utc), 
                               content=[FunctionExecutionResult(content='Here are the total points scored by Miami Heat players in the 2006-2007 season:\n        Udonis Haslem: 844 points\n        Dwayne Wade: 1397 points\n        James Posey: 550 points\n        ...\n        ', name='search_web_tool', call_id='call_d5arZSv1xZvsB0HWjKMJdi8o', is_error=False)], type='ToolCallExecutionEvent'), 
                               
        ToolCallSummaryMessage(id='9c85d85c-b114-412b-95d3-71ea0eaaf5d4', source='WebSearchAgent', models_usage=None, metadata={}, created_at=datetime.datetime(2025, 7, 20, 8, 9, 47, 277218, tzinfo=datetime.timezone.utc), 
                               content='Here are the total points scored by Miami Heat players in the 2006-2007 season:\n        Udonis Haslem: 844 points\n        Dwayne Wade: 1397 points\n        James Posey: 550 points\n        ...\n        ', type='ToolCallSummaryMessage', tool_calls=[FunctionCall(id='call_d5arZSv1xZvsB0HWjKMJdi8o', arguments='{"query":"Miami Heat highest points player 2006-2007 season"}', name='search_web_tool')], results=[FunctionExecutionResult(content='Here are the total points scored by Miami Heat players in the 2006-2007 season:\n        Udonis Haslem: 844 points\n        Dwayne Wade: 1397 points\n        James Posey: 550 points\n        ...\n        ', name='search_web_tool', call_id='call_d5arZSv1xZvsB0HWjKMJdi8o', is_error=False)]), 
                               
        TextMessage(id='fbe94ad1-fe5f-48a4-8d52-4b5899c963b7', 
                    source='PlanningAgent', models_usage=RequestUsage(prompt_tokens=445, completion_tokens=129), metadata={}, created_at=datetime.datetime(2025, 7, 20, 8, 9, 49, 47905, tzinfo=datetime.timezone.utc), 
                    content="Based on the search findings, it appears that Dwyane Wade was the Miami Heat player with the highest points in the 2006-2007 season. Now, let's continue with the next tasks:\n\n3. WebSearchAgent: Find the total rebounds of Dwyane Wade in the 2007-2008 season.\n4. WebSearchAgent: Find the total rebounds of Dwyane Wade in the 2008-2009 season.\n5. DataAnalystAgent: Calculate the percentage change in Dwyane Wade's total rebounds from the 2007-2008 season to the 2008-2009 season.", type='TextMessage'), 
                    
        UserInputRequestedEvent(id='8c194f5a-14bc-4da0-997f-44817da92ccd', 
                                source='UserProxyAgent', models_usage=None, metadata={}, created_at=datetime.datetime(2025, 7, 20, 8, 9, 49, 49905, tzinfo=datetime.timezone.utc), request_id='d8cc3476-dffa-4336-953d-a11eaf4bb4d7', 
                                content='', type='UserInputRequestedEvent'), TextMessage(id='e492b2d1-6867-44b3-8ea1-a0be9b772b5f', source='UserProxyAgent', models_usage=None, metadata={}, created_at=datetime.datetime(2025, 7, 20, 8, 10, 22, 585463, tzinfo=datetime.timezone.utc), content='APPROVE', type='TextMessage'), 
        
        ToolCallRequestEvent(id='b10696c7-91dd-4e4e-a02e-03b5589b176c', 
                             source='WebSearchAgent', models_usage=RequestUsage(prompt_tokens=610, completion_tokens=70), metadata={}, created_at=datetime.datetime(2025, 7, 20, 8, 10, 24, 66012, tzinfo=datetime.timezone.utc), 
                             content=[FunctionCall(id='call_AbGxIAlDfU89yndZAb4ZOqgx', arguments='{"query": "Dwyane Wade total rebounds 2007-2008 season"}', name='search_web_tool'), 
                                      FunctionCall(id='call_Ybjixxq5rBnfRPVY3zU0bfsj', arguments='{"query": "Dwyane Wade total rebounds 2008-2009 season"}', name='search_web_tool')], type='ToolCallRequestEvent'), 
        
        ToolCallExecutionEvent(id='e0afee1e-b0ad-46d0-b67d-c7be2176db00', 
                               source='WebSearchAgent', models_usage=None, metadata={}, created_at=datetime.datetime(2025, 7, 20, 8, 10, 24, 70012, tzinfo=datetime.timezone.utc), 
                               content=[FunctionExecutionResult(content='The number of total rebounds for Dwayne Wade in the Miami Heat season 2007-2008 is 214.', name='search_web_tool', call_id='call_AbGxIAlDfU89yndZAb4ZOqgx', is_error=False), 
                                        FunctionExecutionResult(content='The number of total rebounds for Dwayne Wade in the Miami Heat season 2008-2009 is 398.', name='search_web_tool', call_id='call_Ybjixxq5rBnfRPVY3zU0bfsj', is_error=False)], type='ToolCallExecutionEvent'), 
                                        
        ToolCallSummaryMessage(id='dce97680-4b9a-4283-b8bb-ecfd6e820ee2', 
                               source='WebSearchAgent', models_usage=None, metadata={}, created_at=datetime.datetime(2025, 7, 20, 8, 10, 24, 70012, tzinfo=datetime.timezone.utc), 
                               content='The number of total rebounds for Dwayne Wade in the Miami Heat season 2007-2008 is 214.\nThe number of total rebounds for Dwayne Wade in the Miami Heat season 2008-2009 is 398.', type='ToolCallSummaryMessage', tool_calls=[FunctionCall(id='call_AbGxIAlDfU89yndZAb4ZOqgx', arguments='{"query": "Dwyane Wade total rebounds 2007-2008 season"}', name='search_web_tool'), 
                               FunctionCall(id='call_Ybjixxq5rBnfRPVY3zU0bfsj', arguments='{"query": "Dwyane Wade total rebounds 2008-2009 season"}', name='search_web_tool')], 
                               results=[FunctionExecutionResult(content='The number of total rebounds for Dwayne Wade in the Miami Heat season 2007-2008 is 214.', name='search_web_tool', call_id='call_AbGxIAlDfU89yndZAb4ZOqgx', is_error=False), 
                                        FunctionExecutionResult(content='The number of total rebounds for Dwayne Wade in the Miami Heat season 2008-2009 is 398.', name='search_web_tool', call_id='call_Ybjixxq5rBnfRPVY3zU0bfsj', is_error=False)]), 
                               
        TextMessage(id='640480fc-f63e-4970-8936-74241ddc0dad', 
                    source='PlanningAgent', models_usage=RequestUsage(prompt_tokens=645, completion_tokens=88), metadata={}, created_at=datetime.datetime(2025, 7, 20, 8, 10, 25, 434317, tzinfo=datetime.timezone.utc), 
                    content="Great, we now have the necessary rebound data. Let's move on to calculate the percentage change in Dwyane Wade's total rebounds between the 2007-2008 and 2008-2009 seasons:\n\n5. DataAnalystAgent: Calculate the percentage change in Dwyane Wade's total rebounds from the 2007-2008 season (214 rebounds) to the 2008-2009 season (398 rebounds).", type='TextMessage'), 
                    UserInputRequestedEvent(id='212045ca-d832-4bc7-a208-4d1222d9ffc4', source='UserProxyAgent', models_usage=None, metadata={}, created_at=datetime.datetime(2025, 7, 20, 8, 10, 25, 436320, tzinfo=datetime.timezone.utc), request_id='e9399bcc-164d-4f20-a011-a0d65009f416', content='', type='UserInputRequestedEvent'), 
                    
        TextMessage(id='423ef667-b6ab-4d5b-835e-e082cfab718f', 
                    source='UserProxyAgent', models_usage=None, metadata={}, created_at=datetime.datetime(2025, 7, 20, 8, 10, 28, 957678, tzinfo=datetime.timezone.utc), 
                    content='APPROVE', type='TextMessage'), 
                    
        ToolCallRequestEvent(id='6369c56f-1db8-422f-8f4a-ec6ac7539f25', 
                             source='DataAnalystAgent', models_usage=RequestUsage(prompt_tokens=746, completion_tokens=20), metadata={}, created_at=datetime.datetime(2025, 7, 20, 8, 10, 30, 387506, tzinfo=datetime.timezone.utc), 
                             content=[FunctionCall(id='call_2Wplt08H59hhXMhsa64oHimx', arguments='{"start":214,"end":398}', name='percentage_change_tool')], type='ToolCallRequestEvent'), 
                             
        ToolCallExecutionEvent(id='0ebc5089-04f8-4019-88ed-4a41787ca787', source='DataAnalystAgent', models_usage=None, metadata={}, created_at=datetime.datetime(2025, 7, 20, 8, 10, 30, 391487, tzinfo=datetime.timezone.utc), 
                               content=[FunctionExecutionResult(content='85.98130841121495', name='percentage_change_tool', call_id='call_2Wplt08H59hhXMhsa64oHimx', is_error=False)], type='ToolCallExecutionEvent'), 
                               
        ToolCallSummaryMessage(id='746184a6-63c4-4fe1-9d83-928692b14f10', 
                               source='DataAnalystAgent', models_usage=None, metadata={}, created_at=datetime.datetime(2025, 7, 20, 8, 10, 30, 391487, tzinfo=datetime.timezone.utc), 
                               content='85.98130841121495', type='ToolCallSummaryMessage', tool_calls=[FunctionCall(id='call_2Wplt08H59hhXMhsa64oHimx', arguments='{"start":214,"end":398}', name='percentage_change_tool')], results=[FunctionExecutionResult(content='85.98130841121495', name='percentage_change_tool', call_id='call_2Wplt08H59hhXMhsa64oHimx', is_error=False)]), 
                               
        TextMessage(id='9f9fd44c-68fd-46b4-b225-66ddbe93b68e', 
                    source='PlanningAgent', models_usage=RequestUsage(prompt_tokens=764, completion_tokens=122), metadata={}, created_at=datetime.datetime(2025, 7, 20, 8, 10, 31, 962986, tzinfo=datetime.timezone.utc), 
                    content="The percentage change in Dwyane Wade's total rebounds from the 2007-2008 season to the 2008-2009 season is approximately 85.98%.\n\nSummary:\n- The Miami Heat player with the highest points in the 2006-2007 season was Dwyane Wade.\n- Dwyane Wade had 214 total rebounds in the 2007-2008 season and 398 total rebounds in the 2008-2009 season.\n- The percentage increase in Dwyane Wade's total rebounds between these two seasons was about 85.98%.\n\nTERMINATE", type='TextMessage')], 
           
           
    stop_reason="Text 'TERMINATE' mentioned")